# PyPSA Electricity Network Model - TYNDP 2030

This notebook builds an electricity-only network model for Europe based on TYNDP 2024 data for the year 2030.

## Phase 1: Setup & Data Loading

Import necessary libraries and load all data files from `data/open-tyndp/`.

In [5]:
!python3 -m pip install --upgrade pip
!python3 -m pip install pypsa matplotlib seaborn pandas numpy


In [7]:
!python3 -m pip install pypsa

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import pypsa
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"PyPSA version: {pypsa.__version__}")

/Users/radovanhodor/Library/Python/3.10/lib/python/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Libraries imported successfully
PyPSA version: 0.35.2


In [2]:
# Define data directory
DATA_DIR = Path("../data/open-tyndp")

# Load all CSV files
print("📂 Loading data files...\n")

# Network topology
buses = pd.read_csv(DATA_DIR / "buses.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

# Capacities and costs
capacities = pd.read_csv(DATA_DIR / "pemmdb_capacities_2030.csv")
costs = pd.read_csv(DATA_DIR / "costs_2030_processed.csv")

# Demand
electricity_demand = pd.read_csv(DATA_DIR / "electricity_demand_2030.csv", index_col=0, parse_dates=True)

# Availability profiles for renewables
wind_onshore = pd.read_csv(DATA_DIR / "pecd_data_Wind_Onshore_2030.csv", index_col=0, parse_dates=True)
wind_offshore = pd.read_csv(DATA_DIR / "pecd_data_Wind_Offshore_2030.csv", index_col=0, parse_dates=True)
solar_utility = pd.read_csv(DATA_DIR / "pecd_data_LFSolarPVUtility_2030.csv", index_col=0, parse_dates=True)
solar_rooftop = pd.read_csv(DATA_DIR / "pecd_data_LFSolarPVRooftop_2030.csv", index_col=0, parse_dates=True)

# Availability profile for thermal/synchronous generators
thermal_avail = pd.read_csv(DATA_DIR / "avail_profile_s_all.csv", index_col=0, parse_dates=True)

# Hydro inflows (5 types)
hydro_ror = pd.read_csv(DATA_DIR / "hydro_inflows_tyndp_Run_of_River_2030.csv", index_col=0, parse_dates=True)
hydro_reservoir = pd.read_csv(DATA_DIR / "hydro_inflows_tyndp_Reservoir_2030.csv", index_col=0, parse_dates=True)
hydro_pondage = pd.read_csv(DATA_DIR / "hydro_inflows_tyndp_Pondage_2030.csv", index_col=0, parse_dates=True)
hydro_ps_open = pd.read_csv(DATA_DIR / "hydro_inflows_tyndp_PS_Open_2030.csv", index_col=0, parse_dates=True)
hydro_ps_closed = pd.read_csv(DATA_DIR / "hydro_inflows_tyndp_PS_Closed_2030.csv", index_col=0, parse_dates=True)

# CO2 totals (for validation)
co2_totals = pd.read_csv(DATA_DIR / "co2_totals.csv")

print("✅ All data files loaded successfully!\n")

# Display basic info
print(f"📊 Data Summary:")
print(f"   Buses: {len(buses)} nodes")
print(f"   Links: {len(links)} connections")
print(f"   Capacities: {len(capacities)} rows × {len(capacities.columns)} technologies")
print(f"   Demand timesteps: {len(electricity_demand)}")
print(f"   Wind onshore timesteps: {len(wind_onshore)}")
print(f"   Wind offshore timesteps: {len(wind_offshore)}")
print(f"   Solar utility timesteps: {len(solar_utility)}")
print(f"   Solar rooftop timesteps: {len(solar_rooftop)}")
print(f"   Thermal availability timesteps: {len(thermal_avail)}")

📂 Loading data files...

✅ All data files loaded successfully!

📊 Data Summary:
   Buses: 55 nodes
   Links: 219 connections
   Capacities: 2646 rows × 9 technologies
   Demand timesteps: 8760
   Wind onshore timesteps: 8760
   Wind offshore timesteps: 8760
   Solar utility timesteps: 8760
   Solar rooftop timesteps: 8760
   Thermal availability timesteps: 8760


## Phase 2: Network Topology

Create PyPSA network and add buses (nodes) and transmission lines.

In [3]:
# Create PyPSA network
network = pypsa.Network()
network.name = "TYNDP2024_Electricity_2030"

# Set snapshots - we'll start with a single snapshot, then expand in Phase 6
network.set_snapshots(pd.date_range('2030-01-01', periods=1, freq='h'))

print(f"✅ Created network: {network.name}")
print(f"   Initial snapshots: {len(network.snapshots)}")

✅ Created network: TYNDP2024_Electricity_2030
   Initial snapshots: 1


In [9]:
# Inspect buses data structure
print("📊 Buses data structure:")
display(buses.head())
print(f"\nColumns: {buses.columns.tolist()}")
print(f"\nSample bus codes: {buses['bus_id'].head(10).tolist()}")
print(f"   Total buses: {len(buses)}")
print(f"   Unique bus_ids: {buses['bus_id'].unique()}")

📊 Buses data structure:


,bus_id,station_id,voltage,dc,symbol,under_construction,tags,x,y,country,geometry
0,AL00,AL00,380,NaN,Substation,f,AL00,20.036884,41.117588,AL,POINT (20.036883988642362 41.117587702511265)
1,AT00,AT00,380,NaN,Substation,f,AT00,14.822183,47.668898,AT,POINT (14.822183225330722 47.66889815500067)
2,BA00,BA00,380,NaN,Substation,f,BA00,17.867837,43.982016,BA,POINT (17.867837381251395 43.98201574415204)
3,BE00,BE00,380,NaN,Substation,f,BE00,4.967931,50.470635,BE,POINT (4.96793113501169 50.47063494467691)
4,BG00,BG00,380,NaN,Substation,f,BG00,25.323948,42.668760,BG,POINT (25.323948321769798 42.66875960983357)



Columns: ['bus_id', 'station_id', 'voltage', 'dc', 'symbol', 'under_construction', 'tags', 'x', 'y', 'country', 'geometry']

Sample bus codes: ['AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1']
   Total buses: 55
   Unique bus_ids: ['AL00' 'AT00' 'BA00' 'BE00' 'BG00' 'CH00' 'CY00' 'CZ00' 'DE00' 'DKE1'
 'DKW1' 'EE00' 'ES00' 'FI00' 'FR00' 'FR15' 'GB00' 'GBNI' 'GR00' 'GR03'
 'HR00' 'HU00' 'IE00' 'ITCA' 'ITCN' 'ITCS' 'ITN1' 'ITS1' 'ITSA' 'ITSI'
 'LT00' 'LUB1' 'LUF1' 'LUG1' 'LUV1' 'LV00' 'ME00' 'MK00' 'MT00' 'NL00'
 'NOM1' 'NON1' 'NOS0' 'PL00' 'PT00' 'RO00' 'RS00' 'SE01' 'SE02' 'SE03'
 'SE04' 'SI00' 'SK00' 'ITCO' 'ITVI']


In [11]:
# Add buses to network
for idx, row in buses.iterrows():
    network.add("Bus",
                name=row['bus_id'],
                x=row['x'],
                y=row['y'],
                country=row['country'],
                v_nom=row.get('voltage', 380))  # nominal voltage in kV

print(f"✅ Added {len(network.buses)} buses to network")
print(f"\nSample buses:")
display(network.buses.head())

✅ Added 55 buses to network

Sample buses:


,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country
Bus,,,,,,,,,,,,,,
AL00,380.0,,20.036884,41.117588,AC,,,1.0,0.0,inf,PQ,,,AL
AT00,380.0,,14.822183,47.668898,AC,,,1.0,0.0,inf,PQ,,,AT
BA00,380.0,,17.867837,43.982016,AC,,,1.0,0.0,inf,PQ,,,BA
BE00,380.0,,4.967931,50.470635,AC,,,1.0,0.0,inf,PQ,,,BE
BG00,380.0,,25.323948,42.668760,AC,,,1.0,0.0,inf,PQ,,,BG


In [ ]:
# Add DC links (HVDC) to network
# The index contains the link_id, link_id column contains bus0, bus0 column contains bus1
added_links = 0
skipped_links = 0

for link_id_full, row in links.iterrows():
    bus0 = row['link_id']  # actual bus0
    bus1 = row['bus0']  # actual bus1
    
    # Check if both buses exist in the network
    if bus0 not in network.buses.index or bus1 not in network.buses.index:
        skipped_links += 1
        continue
    
    # Calculate link length from bus coordinates (km)
    x0, y0 = network.buses.loc[bus0, ['x', 'y']]
    x1, y1 = network.buses.loc[bus1, ['x', 'y']]
    # Simple Euclidean distance * 111 km/degree (approximate)
    length_km = np.sqrt((x1-x0)**2 + (y1-y0)**2) * 111
    
    network.add("Link",
                name=link_id_full,
                bus0=bus0,
                bus1=bus1,
                p_nom=row['p_nom'],  # MW capacity
                p_nom_extendable=False,  # do not allow expansion in future phases
                length=length_km,  # km
                carrier='DC',
                efficiency=0.97)  # typical HVDC efficiency (3% losses)
    added_links += 1

print(f"✅ Added {added_links} DC links to network")
if skipped_links > 0:
    print(f"⚠️  Skipped {skipped_links} links (buses not found)")
    
print(f"\nSample links:")
display(network.links[['bus0', 'bus1', 'p_nom', 'length', 'carrier', 'efficiency']].head(10))

✅ Added 219 DC links to network

Sample links:


,bus0,bus1,p_nom,length,carrier,efficiency
Link,,,,,,
AL00-GR00-DC,AL00,GR00,219635.110617,245.452182,DC,0.97
AL00-ME00-DC,AL00,ME00,189342.890082,202.828059,DC,0.97
AL00-MK00-DC,AL00,MK00,153308.706607,196.445037,DC,0.97
AL00-RS00-DC,AL00,RS00,351420.535321,356.327878,DC,0.97
AT00-CH00-DC,AT00,CH00,494893.193612,717.623708,DC,0.97
AT00-CZ00-DC,AT00,CZ00,239946.178070,249.446315,DC,0.97
AT00-DE00-DC,AT00,DE00,512754.700056,647.361612,DC,0.97
AT00-HU00-DC,AT00,HU00,324765.186895,475.066001,DC,0.97
AT00-ITN1-DC,AT00,ITN1,464387.903281,619.653116,DC,0.97


## Phase 3: Add Generators

Add power generation capacity for all technologies from TYNDP 2030 capacities data.

In [14]:
# Inspect capacities data structure
print("📊 Capacities data structure:")
display(capacities.head())
print(f"\nShape: {capacities.shape}")
print(f"\nColumns: {capacities.columns.tolist()}")

📊 Capacities data structure:


,carrier,index_carrier,bus,p_nom,e_nom,efficiency,country,unit,open_tyndp_type
0,electrolyser,electrolyser-onshore-grid,AL00,0.000,0.0,0.0,AL,MW,electrolyser-onshore-grid
1,hydro-ror,hydro-ror-turbine,AL00,603.672,0.0,1.0,AL,MW,hydro-ror-turbine
2,hydro-pondage,hydro-pondage-reservoir,AL00,0.000,0.0,1.0,AL,MWh,hydro-pondage-reservoir
3,hydro-pondage,hydro-pondage-turbine,AL00,0.000,0.0,1.0,AL,MW,hydro-pondage-turbine
4,hydro-reservoir,hydro-reservoir-reservoir,AL00,0.000,1721680.0,1.0,AL,MWh,hydro-reservoir-reservoir



Shape: (2646, 9)

Columns: ['carrier', 'index_carrier', 'bus', 'p_nom', 'e_nom', 'efficiency', 'country', 'unit', 'open_tyndp_type']


In [15]:
# Check unique carriers (technologies) in capacities
print("📋 Available carriers (technologies):")
carrier_counts = capacities.groupby('carrier').agg({
    'p_nom': lambda x: (x > 0).sum(),
    'bus': 'count'
}).rename(columns={'p_nom': 'buses_with_capacity', 'bus': 'total_entries'})
display(carrier_counts)

print("\n📋 Unique carriers:")
print(capacities['carrier'].unique())

📋 Available carriers (technologies):


,buses_with_capacity,total_entries
carrier,,
battery,70,159
chp,50,50
coal,19,212
electrolyser,34,52
gas,147,530
h2-ccgt,5,53
h2-fuel-cell,1,53
hydro-phs,20,159
hydro-phs-pure,21,159



📋 Unique carriers:
['electrolyser' 'hydro-ror' 'hydro-pondage' 'hydro-reservoir' 'hydro-phs'
 'hydro-phs-pure' 'onwind' 'offwind' 'solar-thermal' 'solar-pv-utility'
 'solar-pv-rooftop' 'solar-thermal-w-storage' 'battery' 'uranium' 'coal'
 'lignite' 'gas' 'oil-light' 'oil-heavy' 'oil-shale' 'h2-fuel-cell'
 'h2-ccgt' 'other-res' 'chp']


In [17]:
# Inspect costs data structure
print("📊 Costs data structure:")
display(costs.head(10))
print(f"\nShape: {costs.shape}")
print(f"\nColumns: {costs.columns.tolist()}")
print("\n📋 Unique carriers in costs:")
print(costs['technology'].unique())

📊 Costs data structure:


,technology,Bottom storage temperature,C in fuel,C stored,CO2 intensity,CO2 stored,FOM,Motor size,Top storage temperature,VOM,...,pelletizing cost,scrap-input,slag-input,standing_losses,temperature difference,wood-input,yield-biochar,capital_cost,marginal_cost,CO2 capture
0,Alkaline electrolyzer large size,NaN,NaN,NaN,0.0,NaN,2.8000,NaN,NaN,0.2389,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,44194.072656,0.2389,NaN
1,Alkaline electrolyzer medium size,NaN,NaN,NaN,0.0,NaN,2.3000,NaN,NaN,0.2389,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,59404.717871,0.2389,NaN
2,Alkaline electrolyzer small size,NaN,NaN,NaN,0.0,NaN,2.3000,NaN,NaN,0.1934,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,68430.919060,0.1934,NaN
3,Ammonia cracker,NaN,NaN,NaN,0.0,NaN,4.3000,NaN,NaN,0.0000,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,144775.985816,0.0000,NaN
4,BEV Bus city,NaN,NaN,NaN,0.0,NaN,0.0003,346.5517,NaN,0.0952,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,28012.052635,0.0952,NaN
5,BEV Coach,NaN,NaN,NaN,0.0,NaN,0.0002,358.6207,NaN,0.0952,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,38152.112668,0.0952,NaN
6,BEV Truck Semi-Trailer max 50 tons,NaN,NaN,NaN,0.0,NaN,0.0004,555.1724,NaN,0.0952,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,20814.213321,0.0952,NaN
7,BEV Truck Solo max 26 tons,NaN,NaN,NaN,0.0,NaN,0.0002,382.7586,NaN,0.0952,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,32574.846501,0.0952,NaN
8,BEV Truck Trailer max 56 tons,NaN,NaN,NaN,0.0,NaN,0.0003,710.3448,NaN,0.0952,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,19345.717684,0.0952,NaN
9,Battery electric (passenger cars),NaN,NaN,NaN,0.0,NaN,0.9000,NaN,NaN,0.0000,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2925.198839,0.0000,NaN



Shape: (307, 63)

Columns: ['technology', 'Bottom storage temperature', 'C in fuel', 'C stored', 'CO2 intensity', 'CO2 stored', 'FOM', 'Motor size', 'Top storage temperature', 'VOM', 'ammonia-input', 'c_b', 'c_v', 'capacity', 'capture rate', 'capture_rate', 'carbondioxide-input', 'carbondioxide-output', 'clinker-input', 'coal-input', 'commodity', 'compression-electricity-input', 'compression-heat-output', 'discount rate', 'district heat surcharge', 'district heat-input', 'economic_lifetime', 'efficiency', 'efficiency-biochar', 'efficiency-biomass', 'efficiency-electricity', 'efficiency-heat', 'efficiency-hydrogen', 'efficiency-tot', 'electricity-input', 'energy to power ratio', 'fuel', 'gas-input', 'hbi-input', 'heat-input', 'heat-output', 'hydrogen-input', 'investment', 'lifetime', 'lohc-input', 'methane-input', 'methanol-input', 'min_fill_level', 'naphtha-input', 'nitrogen-input', 'oil-input', 'ore-input', 'p_nom_ratio', 'pelletizing cost', 'scrap-input', 'slag-input', 'standing_los

In [20]:
# Create a mapping from carrier names to cost data
# We need to map TYNDP carrier names to technology names in costs
carrier_to_tech_mapping = {
    'onwind': 'onshore wind',
    'offwind': 'offshore wind',
    'solar-pv-utility': 'solar-utility',
    'solar-pv-rooftop': 'solar-rooftop',
    'solar-thermal': 'solar thermal',
    'hydro-ror': 'hydro',
    'hydro-reservoir': 'hydro',
    'hydro-pondage': 'hydro',
    'uranium': 'nuclear',
    'coal': 'coal',
    'lignite': 'lignite',
    'gas': 'CCGT',
    'oil-light': 'oil',
    'oil-heavy': 'oil',
    'h2-fuel-cell': 'fuel cell',
    'h2-ccgt': 'H2 turbine',
    'chp': 'CCGT',
    'other-res': 'biomass CHP'
}

# Create costs dictionary with default values
cost_dict = {}
for tech in costs['technology'].unique():
    cost_dict[tech] = {
        'capital_cost': costs[costs['technology'] == tech]['capital_cost'].iloc[0] if len(costs[costs['technology'] == tech]) > 0 else 0,
        'marginal_cost': costs[costs['technology'] == tech]['marginal_cost'].iloc[0] if len(costs[costs['technology'] == tech]) > 0 else 0,
        'efficiency': costs[costs['technology'] == tech]['efficiency'].iloc[0] if 'efficiency' in costs.columns and len(costs[costs['technology'] == tech]) > 0 else 1.0,
        'lifetime': costs[costs['technology'] == tech]['lifetime'].iloc[0] if 'lifetime' in costs.columns and len(costs[costs['technology'] == tech]) > 0 else 25
    }

print(f"✅ Created cost dictionary with {len(cost_dict)} technologies")
print(f"\nSample cost mapping for renewables:")
for carrier in ['onwind', 'offwind', 'solar-pv-utility', 'uranium', 'gas']:
    tech = carrier_to_tech_mapping.get(carrier, carrier)
    if tech in cost_dict:
        print(f"  {carrier} -> {tech}: capital={cost_dict[tech]['capital_cost']:.0f} €/MW, marginal={cost_dict[tech]['marginal_cost']:.2f} €/MWh")


display(pd.DataFrame.from_dict(cost_dict, orient='index').head(10))

✅ Created cost dictionary with 307 technologies

Sample cost mapping for renewables:
  solar-pv-utility -> solar-utility: capital=38283 €/MW, marginal=0.00 €/MWh
  uranium -> nuclear: capital=245734 €/MW, marginal=1.16 €/MWh
  gas -> CCGT: capital=104788 €/MW, marginal=46.80 €/MWh


,capital_cost,marginal_cost,efficiency,lifetime
Alkaline electrolyzer large size,44194.072656,0.2389,1.0000,40.0
Alkaline electrolyzer medium size,59404.717871,0.2389,1.0000,20.0
Alkaline electrolyzer small size,68430.919060,0.1934,1.0000,20.0
Ammonia cracker,144775.985816,0.0000,1.0000,25.0
BEV Bus city,28012.052635,0.0952,0.8585,12.0
BEV Coach,38152.112668,0.0952,0.8446,12.0
BEV Truck Semi-Trailer max 50 tons,20814.213321,0.0952,1.3936,10.5
BEV Truck Solo max 26 tons,32574.846501,0.0952,0.8755,13.8
BEV Truck Trailer max 56 tons,19345.717684,0.0952,1.5446,13.8
Battery electric (passenger cars),2925.198839,0.0000,0.6800,15.0


In [ ]:
# Define which carriers are generators (not storage or other components)
generator_carriers = [
    'onwind', 'offwind', 
    'solar-pv-utility', 'solar-pv-rooftop', 'solar-thermal',
    'hydro-ror',  # run-of-river (no storage)
    'uranium',  # nuclear
    'coal', 'lignite',
    'gas',  # CCGT/OCGT
    'oil-light', 'oil-heavy', 'oil-shale',
    'h2-fuel-cell', 'h2-ccgt',
    'chp',
    'other-res'  # other renewables (biomass, etc.)
]

# Filter capacities for generators only (p_nom > 0 and unit == 'MW')
gen_capacities = capacities[
    (capacities['carrier'].isin(generator_carriers)) &
    (capacities['unit'] == 'MW') &
    (capacities['p_nom'] > 0)
].copy()

print(f"📊 Found {len(gen_capacities)} generator entries with capacity > 0")
print(f"\nBy carrier:")
print(gen_capacities.groupby('carrier')['p_nom'].agg(['count', 'sum']).sort_values('sum', ascending=False))

In [ ]:
# Add generators to network
added_generators = 0
skipped_generators = 0

for idx, row in gen_capacities.iterrows():
    bus = row['bus']
    carrier = row['carrier']
    p_nom = row['p_nom']
    
    # Check if bus exists
    if bus not in network.buses.index:
        skipped_generators += 1
        continue
    
    # Get cost data
    tech_name = carrier_to_tech_mapping.get(carrier, carrier)
    costs_data = cost_dict.get(tech_name, {})
    
    # Use efficiency from capacities data if available, otherwise from costs
    efficiency = row.get('efficiency', costs_data.get('efficiency', 1.0))
    if pd.isna(efficiency) or efficiency == 0:
        efficiency = 1.0
    
    # Create unique generator name
    gen_name = f"{bus}-{carrier}"
    
    # Add generator
    network.add("Generator",
                name=gen_name,
                bus=bus,
                carrier=carrier,
                p_nom=p_nom,
                p_nom_extendable=False,  # allow expansion in future phases
                efficiency=efficiency,
                capital_cost=costs_data.get('capital_cost', 0),
                marginal_cost=costs_data.get('marginal_cost', 0))
    
    added_generators += 1

print(f"✅ Added {added_generators} generators to network")
if skipped_generators > 0:
    print(f"⚠️  Skipped {skipped_generators} generators (bus not found)")

print(f"\n📊 Generators by carrier:")
gen_by_carrier = network.generators.groupby('carrier')['p_nom'].agg(['count', 'sum']).sort_values('sum', ascending=False)
print(gen_by_carrier)

print(f"\n📊 Total installed capacity: {network.generators['p_nom'].sum():.0f} MW")

## Phase 4: Add Storage Units

Add energy storage systems: batteries, pumped hydro storage, and hydro reservoirs.

In [ ]:
# Identify storage carriers in capacities data
storage_carriers = ['battery', 'hydro-phs', 'hydro-phs-pure', 'hydro-reservoir', 'hydro-pondage']

# Filter for storage entries
storage_capacities = capacities[capacities['carrier'].isin(storage_carriers)].copy()

print(f"📊 Storage entries in capacities:")
print(f"Total entries: {len(storage_capacities)}")
print(f"\nBy carrier:")
print(storage_capacities.groupby('carrier').agg({
    'p_nom': lambda x: (x > 0).sum(),
    'e_nom': lambda x: (x > 0).sum()
}).rename(columns={'p_nom': 'p_nom_entries', 'e_nom': 'e_nom_entries'}))

print(f"\nSample storage data:")
print(storage_capacities[storage_capacities['p_nom'] > 0].head(10))

In [ ]:
# Group storage data by bus and carrier to aggregate charge/discharge/storage capacities
# For each storage unit, we need: p_nom (power), e_nom (energy), efficiency

storage_grouped = {}

for bus in storage_capacities['bus'].unique():
    bus_storage = storage_capacities[storage_capacities['bus'] == bus]
    
    for carrier in bus_storage['carrier'].unique():
        carrier_data = bus_storage[bus_storage['carrier'] == carrier]
        
        # Extract power and energy capacities
        p_nom_values = carrier_data[carrier_data['p_nom'] > 0]['p_nom'].values
        e_nom_values = carrier_data[carrier_data['e_nom'] > 0]['e_nom'].values
        efficiency_values = carrier_data[carrier_data['efficiency'] > 0]['efficiency'].values
        
        if len(p_nom_values) > 0 or len(e_nom_values) > 0:
            key = f"{bus}-{carrier}"
            storage_grouped[key] = {
                'bus': bus,
                'carrier': carrier,
                'p_nom': p_nom_values[0] if len(p_nom_values) > 0 else 0,
                'e_nom': e_nom_values[0] if len(e_nom_values) > 0 else 0,
                'efficiency': efficiency_values[0] if len(efficiency_values) > 0 else 0.9
            }

print(f"📊 Grouped storage units: {len(storage_grouped)}")
print(f"\nSample grouped storage:")
for i, (key, data) in enumerate(list(storage_grouped.items())[:5]):
    print(f"  {key}: p_nom={data['p_nom']:.1f} MW, e_nom={data['e_nom']:.1f} MWh, eff={data['efficiency']:.2f}")

In [ ]:
# Add storage units to network
added_storage = 0
skipped_storage = 0

for key, data in storage_grouped.items():
    bus = data['bus']
    carrier = data['carrier']
    p_nom = data['p_nom']
    e_nom = data['e_nom']
    efficiency = data['efficiency']
    
    # Check if bus exists
    if bus not in network.buses.index:
        skipped_storage += 1
        continue
    
    # Skip if no capacity
    if p_nom <= 0 and e_nom <= 0:
        skipped_storage += 1
        continue
    
    # Get cost data
    tech_name = carrier_to_tech_mapping.get(carrier, carrier)
    if carrier == 'battery':
        tech_name = 'battery storage'
    elif 'hydro-phs' in carrier:
        tech_name = 'PHS'
    elif 'hydro-reservoir' in carrier or 'hydro-pondage' in carrier:
        tech_name = 'hydro'
    
    costs_data = cost_dict.get(tech_name, {})
    
    # Calculate max_hours (storage duration)
    if p_nom > 0 and e_nom > 0:
        max_hours = e_nom / p_nom
    elif e_nom > 0:
        max_hours = 6  # default 6 hours
        p_nom = e_nom / max_hours
    elif p_nom > 0:
        max_hours = 6  # default 6 hours
    else:
        skipped_storage += 1
        continue
    
    # Set efficiency values
    if carrier == 'battery':
        efficiency_store = 0.92  # battery charge efficiency
        efficiency_dispatch = 0.92  # battery discharge efficiency
    elif 'hydro-phs' in carrier:
        efficiency_store = 0.87  # pumping efficiency
        efficiency_dispatch = 0.90  # turbine efficiency
    else:
        efficiency_store = 0.95
        efficiency_dispatch = 0.95
    
    # Add StorageUnit
    network.add("StorageUnit",
                name=key,
                bus=bus,
                carrier=carrier,
                p_nom=p_nom,
                p_nom_extendable=False,  # allow expansion in future phases
                max_hours=max_hours,
                efficiency_store=efficiency_store,
                efficiency_dispatch=efficiency_dispatch,
                cyclic_state_of_charge=True,  # storage level at end = level at start
                capital_cost=costs_data.get('capital_cost', 0),
                marginal_cost=costs_data.get('marginal_cost', 0))
    
    added_storage += 1

print(f"✅ Added {added_storage} storage units to network")
if skipped_storage > 0:
    print(f"⚠️  Skipped {skipped_storage} storage units (bus not found or no capacity)")

print(f"\n📊 Storage by carrier:")
storage_by_carrier = network.storage_units.groupby('carrier')['p_nom'].agg(['count', 'sum']).sort_values('sum', ascending=False)
print(storage_by_carrier)

print(f"\n📊 Total storage power capacity: {network.storage_units['p_nom'].sum():.0f} MW")
print(f"📊 Total storage energy capacity: {(network.storage_units['p_nom'] * network.storage_units['max_hours']).sum():.0f} MWh")

## Phase 5: Add Load (Demand)

Add electricity demand for each bus based on TYNDP 2030 demand profiles.

In [ ]:
# Inspect electricity demand data
print("📊 Electricity demand data structure:")
print(electricity_demand.head(10))
print(f"\nShape: {electricity_demand.shape}")
print(f"\nColumns: {electricity_demand.columns.tolist()}")
print(f"\nIndex (datetime): {electricity_demand.index[:5]}")
print(f"\nCountries in demand data: {electricity_demand.columns.tolist()}")

In [ ]:
# Add load for each bus
# For now, we add load with a single value (we'll add time series in Phase 6)
added_loads = 0
skipped_loads = 0

# Get mean demand for each bus (as placeholder for single snapshot)
mean_demand = electricity_demand.mean()

for bus in network.buses.index:
    # Check if bus has demand data
    if bus in electricity_demand.columns:
        demand_value = mean_demand[bus]
        
        network.add("Load",
                    name=f"{bus}-load",
                    bus=bus,
                    p_set=demand_value)  # MW
        added_loads += 1
    else:
        # Add minimal load if no data available
        network.add("Load",
                    name=f"{bus}-load",
                    bus=bus,
                    p_set=100)  # 100 MW minimum load
        skipped_loads += 1

print(f"✅ Added {added_loads} loads with demand data")
if skipped_loads > 0:
    print(f"⚠️  Added {skipped_loads} loads with default demand (no data available)")

print(f"\n📊 Total mean load: {network.loads['p_set'].sum():.0f} MW")
print(f"\nSample loads:")
print(network.loads[['bus', 'p_set']].head(10))

In [ ]:
# Summary of network so far
print("=" * 60)
print("NETWORK SUMMARY (Static - Single Snapshot)")
print("=" * 60)
print(f"\n📍 Buses: {len(network.buses)}")
print(f"🔗 Links (HVDC): {len(network.links)}")
print(f"⚡ Generators: {len(network.generators)} ({network.generators['p_nom'].sum():.0f} MW)")
print(f"🔋 Storage Units: {len(network.storage_units)} ({network.storage_units['p_nom'].sum():.0f} MW)")
print(f"🏭 Loads: {len(network.loads)} ({network.loads['p_set'].sum():.0f} MW)")
print(f"\n📊 Generation capacity by type:")
print(network.generators.groupby('carrier')['p_nom'].sum().sort_values(ascending=False).head(10))
print(f"\n⚠️  Note: Currently using single snapshot. Time series will be added in Phase 6.")

## Phase 6: Apply Time Series

Expand network to full year (8760 hours) and apply time-varying profiles for generators, loads, and storage.

In [ ]:
# Step 1: Expand snapshots to full year 2030 (8760 hours)
print("⏰ Expanding network snapshots to full year 2030...")

# Create 8760 hourly snapshots for 2030
new_snapshots = pd.date_range('2030-01-01', periods=8760, freq='h')
network.set_snapshots(new_snapshots)

print(f"✅ Network now has {len(network.snapshots)} snapshots")
print(f"   Start: {network.snapshots[0]}")
print(f"   End: {network.snapshots[-1]}")

In [ ]:
# Step 2: Apply load time series
print("📊 Applying load time series...")

# electricity_demand has index starting from 2009, we need to shift to 2030
# We'll use the 2009 profile as a proxy for 2030 (common practice in energy modeling)
demand_2030 = electricity_demand.copy()
demand_2030.index = pd.date_range('2030-01-01', periods=len(demand_2030), freq='h')

applied_loads = 0
for load_name in network.loads.index:
    bus = network.loads.loc[load_name, 'bus']
    
    if bus in demand_2030.columns:
        # Apply time series
        network.loads_t.p_set[load_name] = demand_2030[bus].values
        applied_loads += 1
    else:
        # Keep constant load if no data
        network.loads_t.p_set[load_name] = network.loads.loc[load_name, 'p_set']

print(f"✅ Applied time series to {applied_loads} loads")
print(f"\nSample load profile for DE00:")
print(network.loads_t.p_set['DE00-load'].head())

In [ ]:
# Step 3: Apply availability profiles for renewable generators
print("🌬️☀️ Applying renewable availability profiles...")

# Shift renewable profiles to 2030
wind_onshore_2030 = wind_onshore.copy()
wind_onshore_2030.index = pd.date_range('2030-01-01', periods=len(wind_onshore_2030), freq='h')

wind_offshore_2030 = wind_offshore.copy()
wind_offshore_2030.index = pd.date_range('2030-01-01', periods=len(wind_offshore_2030), freq='h')

solar_utility_2030 = solar_utility.copy()
solar_utility_2030.index = pd.date_range('2030-01-01', periods=len(solar_utility_2030), freq='h')

solar_rooftop_2030 = solar_rooftop.copy()
solar_rooftop_2030.index = pd.date_range('2030-01-01', periods=len(solar_rooftop_2030), freq='h')

# Apply profiles to generators
applied_renewables = 0
for gen_name in network.generators.index:
    bus = network.generators.loc[gen_name, 'bus']
    carrier = network.generators.loc[gen_name, 'carrier']
    
    if carrier == 'onwind' and bus in wind_onshore_2030.columns:
        network.generators_t.p_max_pu[gen_name] = wind_onshore_2030[bus].values
        applied_renewables += 1
    elif carrier == 'offwind' and bus in wind_offshore_2030.columns:
        network.generators_t.p_max_pu[gen_name] = wind_offshore_2030[bus].values
        applied_renewables += 1
    elif carrier == 'solar-pv-utility' and bus in solar_utility_2030.columns:
        network.generators_t.p_max_pu[gen_name] = solar_utility_2030[bus].values
        applied_renewables += 1
    elif carrier == 'solar-pv-rooftop' and bus in solar_rooftop_2030.columns:
        network.generators_t.p_max_pu[gen_name] = solar_rooftop_2030[bus].values
        applied_renewables += 1
    else:
        # For non-variable renewables and thermal, set constant availability
        network.generators_t.p_max_pu[gen_name] = 1.0

print(f"✅ Applied availability profiles to {applied_renewables} renewable generators")
print(f"\nSample wind profile for DE00:")
print(network.generators_t.p_max_pu['DE00-onwind'].head())

In [ ]:
# Step 4: Apply thermal generator availability
print("🔥 Applying thermal generator availability...")

# Shift thermal availability to 2030
thermal_avail_2030 = thermal_avail.copy()
thermal_avail_2030.index = pd.date_range('2030-01-01', periods=len(thermal_avail_2030), freq='h')

# Apply to thermal generators (gas, coal, nuclear, etc.)
thermal_carriers = ['uranium', 'coal', 'lignite', 'gas', 'oil-light', 'oil-heavy', 'oil-shale', 'chp']
applied_thermal = 0

for gen_name in network.generators.index:
    bus = network.generators.loc[gen_name, 'bus']
    carrier = network.generators.loc[gen_name, 'carrier']
    
    if carrier in thermal_carriers:
        if bus in thermal_avail_2030.columns:
            # Apply thermal availability profile
            network.generators_t.p_max_pu[gen_name] = thermal_avail_2030[bus].values
            applied_thermal += 1
        else:
            # Default thermal availability (95%)
            network.generators_t.p_max_pu[gen_name] = 0.95

print(f"✅ Applied thermal availability to {applied_thermal} thermal generators")
print(f"\nSample thermal availability for DE00 (uranium):")
if 'DE00-uranium' in network.generators.index:
    print(network.generators_t.p_max_pu['DE00-uranium'].head())

In [ ]:
# Step 5: Apply hydro run-of-river inflows
print("💧 Applying hydro run-of-river inflows...")

# Shift hydro RoR to 2030
hydro_ror_2030 = hydro_ror.copy()
hydro_ror_2030.index = pd.date_range('2030-01-01', periods=len(hydro_ror_2030), freq='h')

applied_hydro_ror = 0
for gen_name in network.generators.index:
    bus = network.generators.loc[gen_name, 'bus']
    carrier = network.generators.loc[gen_name, 'carrier']
    
    if carrier == 'hydro-ror':
        country = network.buses.loc[bus, 'country']
        if country in hydro_ror_2030.columns:
            # Apply hydro RoR profile (normalized by capacity)
            # hydro_ror_2030 is in MW, we need it as p.u. (fraction of capacity)
            capacity = network.generators.loc[gen_name, 'p_nom']
            if capacity > 0:
                network.generators_t.p_max_pu[gen_name] = hydro_ror_2030[country].values / capacity
                # Clip to [0, 1] range
                network.generators_t.p_max_pu[gen_name] = network.generators_t.p_max_pu[gen_name].clip(0, 1)
                applied_hydro_ror += 1
        else:
            # Default constant availability
            network.generators_t.p_max_pu[gen_name] = 0.5

print(f"✅ Applied hydro RoR inflows to {applied_hydro_ror} generators")

In [ ]:
# Step 6: Apply hydro reservoir inflows for storage units
print("🌊 Applying hydro reservoir and PHS inflows to storage...")

# Shift hydro inflows to 2030
hydro_reservoir_2030 = hydro_reservoir.copy()
hydro_reservoir_2030.index = pd.date_range('2030-01-01', periods=len(hydro_reservoir_2030), freq='h')

hydro_pondage_2030 = hydro_pondage.copy()
hydro_pondage_2030.index = pd.date_range('2030-01-01', periods=len(hydro_pondage_2030), freq='h')

hydro_ps_open_2030 = hydro_ps_open.copy()
hydro_ps_open_2030.index = pd.date_range('2030-01-01', periods=len(hydro_ps_open_2030), freq='h')

applied_hydro_storage = 0
for storage_name in network.storage_units.index:
    bus = network.storage_units.loc[storage_name, 'bus']
    carrier = network.storage_units.loc[storage_name, 'carrier']
    country = network.buses.loc[bus, 'country']
    
    if carrier == 'hydro-reservoir' and country in hydro_reservoir_2030.columns:
        # Apply reservoir inflows
        network.storage_units_t.inflow[storage_name] = hydro_reservoir_2030[country].values
        applied_hydro_storage += 1
    elif carrier == 'hydro-pondage' and country in hydro_pondage_2030.columns:
        network.storage_units_t.inflow[storage_name] = hydro_pondage_2030[country].values
        applied_hydro_storage += 1
    elif carrier in ['hydro-phs', 'hydro-phs-pure'] and country in hydro_ps_open_2030.columns:
        network.storage_units_t.inflow[storage_name] = hydro_ps_open_2030[country].values
        applied_hydro_storage += 1
    else:
        # No natural inflow for batteries and unknown storage
        network.storage_units_t.inflow[storage_name] = 0.0

print(f"✅ Applied inflows to {applied_hydro_storage} hydro storage units")

In [ ]:
# Summary of time series application
print("=" * 60)
print("TIME SERIES APPLICATION SUMMARY")
print("=" * 60)
print(f"\n✅ Network now has {len(network.snapshots)} hourly snapshots for 2030")
print(f"\n📊 Time series applied:")
print(f"   Loads: {len([l for l in network.loads.index if l in network.loads_t.p_set.columns])} / {len(network.loads)}")
print(f"   Generators (p_max_pu): {len([g for g in network.generators.index if g in network.generators_t.p_max_pu.columns])} / {len(network.generators)}")
print(f"   Storage (inflow): {len([s for s in network.storage_units.index if s in network.storage_units_t.inflow.columns])} / {len(network.storage_units)}")

print(f"\n📈 Sample data validation:")
print(f"   Total load (mean): {network.loads_t.p_set.sum(axis=1).mean():.0f} MW")
print(f"   Total load (max): {network.loads_t.p_set.sum(axis=1).max():.0f} MW")
print(f"   Total load (min): {network.loads_t.p_set.sum(axis=1).min():.0f} MW")

print(f"\n⚠️  Network is now ready for optimization!")

In [ ]:
# Save the network before optimization
output_file = "tyndp2024_network_2030_preoptimization.nc"
network.export_to_netcdf(output_file)
print(f"💾 Network saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")